# Lesson: GPU vs CPU Benchmark for Image Training

This notebook demonstrates a workload that favors the GPU DirectML setup over CPU.

It uses the Oxford-IIIT Pets dataset and a simple convolutional model to compare runtime.

The dataset is cached locally in `data/` so future runs reuse the same files.

In [ ]:
from pathlib import Path
import time

import pandas as pd
import torch
import torch_directml

from fastai.vision.all import *

## Local Dataset Cache

We save the Pets dataset into `data/` so it does not need to be downloaded again on future runs.

In [ ]:
local_data_path = Path('data')
local_data_path.mkdir(parents=True, exist_ok=True)

path = untar_data(URLs.PETS, dest=local_data_path)

print('Dataset path:', path)
print('Images folder exists:', (path/'images').exists())

## Build the Image DataLoaders

We use the same dataset split for both the GPU and CPU benchmarks.

In [ ]:
img_path = path/'images'
pat = r'([^/]+)_\\d+.jpg$'
label_func = lambda o: o.name.split('_')[0]

dls_base = ImageDataLoaders.from_name_re(
    img_path, get_image_files(img_path), pat, label_func,
    item_tfms=Resize(128), bs=32,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    num_workers=0
)

dls_base.c = len(dls_base.vocab)
dls_base.show_batch(max_n=6, figsize=(8,8))

## GPU DirectML Benchmark

Train the same ResNet model with the DirectML Adreno device and record the runtime.

In [ ]:
gpu_device = torch_directml.device()
print('DirectML device:', torch_directml.device_name(0))

dls_gpu = dls_base.new(device=gpu_device)
learn_gpu = cnn_learner(dls_gpu, resnet18, metrics=accuracy).to_fp16(enabled=False)
learn_gpu.to(gpu_device)

n_epochs = 2
start = time.perf_counter()
learn_gpu.fit_one_cycle(n_epochs, lr_max=3e-3)
gpu_time = time.perf_counter() - start

print(f'GPU DirectML elapsed: {gpu_time:.2f}s')

## CPU-only Benchmark

Run the same model on CPU to compare the total training time on the same dataset.

In [ ]:
cpu_device = torch.device('cpu')
dls_cpu = dls_base.new(device=cpu_device)
learn_cpu = cnn_learner(dls_cpu, resnet18, metrics=accuracy).to_fp16(enabled=False)
learn_cpu.to(cpu_device)

start = time.perf_counter()
learn_cpu.fit_one_cycle(n_epochs, lr_max=3e-3)
cpu_time = time.perf_counter() - start

print(f'CPU elapsed: {cpu_time:.2f}s')

In [ ]:
results = pd.DataFrame([
    {
        'setup': 'DirectML Adreno GPU',
        'duration_s': gpu_time,
        'duration_min': gpu_time/60.0,
        'notes': 'GPU path with DirectML'
    },
    {
        'setup': 'CPU-only',
        'duration_s': cpu_time,
        'duration_min': cpu_time/60.0,
        'notes': 'CPU baseline'
    }
])

results

## Benchmark Summary

If the GPU setup is faster, this notebook has demonstrated a workload where the DirectML Adreno path is beneficial.

If the CPU path is still faster, the lesson shows that small or unsupported workloads can remain CPU-bound, and a larger GPU-friendly workload should be chosen.